In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, desc, row_number, to_timestamp
from pyspark.sql.types import *
from pyspark.sql import Window, Row
from datetime import datetime
from delta.tables import DeltaTable
import json

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 7, Finished, Available, Finished, False)

In [6]:
BASE_PATH = "abfss://a716f785-25f3-4ec0-8cc2-86fd903cb1d9@onelake.dfs.fabric.microsoft.com/a9fca358-1206-44db-b167-4819cf5831cf/Files"
DQ_LOG_PATH = f"{BASE_PATH}/silver/dq_log"

# Bronze Shortcut Paths
BRONZE_PATHS = {
    "customers":        f"{BASE_PATH}/bronze/mysql/customers",
    "subscriptions":    f"{BASE_PATH}/bronze/mysql/subscriptions",
    "fx_rates":         f"{BASE_PATH}/bronze/mysql/fx_rates",

    "towers":           f"{BASE_PATH}/bronze/postgres/towers",
    "network_events":   f"{BASE_PATH}/bronze/postgres/network_events",
    "vendors":          f"{BASE_PATH}/bronze/postgres/vendors",

    "support_tickets":  f"{BASE_PATH}/bronze/gsheets/support_tickets"
}

# Watermark Paths
WATERMARK_PATHS = {
    table: f"{BASE_PATH}/watermarks/{table}.txt"
    for table in BRONZE_PATHS.keys()
}

# Silver Base Path
SILVER_BASE = f"{BASE_PATH}/silver"


StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 8, Finished, Available, Finished, False)

In [7]:
def get_watermark(path):
    try:
        return mssparkutils.fs.head(path, 1024).strip()
    except:
        return "1900-01-01 00:00:00"


def update_watermark(path, ts):
    if ts:
        mssparkutils.fs.put(
            path,
            ts.strftime("%Y-%m-%d %H:%M:%S"),
            overwrite=True
        )

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 9, Finished, Available, Finished, False)

In [8]:
def read_incremental(table, timestamp_col):

    base_path = BRONZE_PATHS[table]
    wm_path   = WATERMARK_PATHS[table]

    last_wm_str = get_watermark(wm_path)
    last_wm = to_timestamp(lit(last_wm_str))

    df = spark.read.parquet(base_path)

    df = df.withColumn(
        timestamp_col,
        col(timestamp_col).cast("timestamp")
    )

    df_incr = df.filter(col(timestamp_col) > last_wm)

    latest_ts = None
    if df_incr.count() > 0:
        latest_ts = df_incr.agg(F.max(timestamp_col)).collect()[0][0]

    return df_incr, latest_ts

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 10, Finished, Available, Finished, False)

In [ ]:
def build_dq_log(table, total, passed, failure_reasons, latest_ts):

    rows_failed = total - passed

    pass_rate = round((passed / total) * 100, 2) if total > 0 else 0

    dq_log = {
        "table": table,
        "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_rows_read": total,
        "rows_passed": passed,
        "rows_failed": rows_failed,
        "pass_rate_pct": pass_rate,
        "failure_reasons": failure_reasons,
        "max_source_timestamp": str(latest_ts) if latest_ts else None
    }

    return dq_log


def write_dq_log(table, dq_log):

    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    file_path = f"{DQ_LOG_PATH}/{table}_{timestamp_str}.json"

    mssparkutils.fs.put(
        file_path,
        json.dumps(dq_log, indent=2),
        overwrite=True
    )

    print(f"  ✅ DQ log written → {file_path}")


In [9]:
def write_silver(df, table, primary_key):

    path = f"{SILVER_BASE}/{table}"

    #  auto partition columns
    if "created_at" in df.columns:
        df = (df
            .withColumn("year", F.year("created_at"))
            .withColumn("month", F.month("created_at"))
        )

    partition_cols = [c for c in ["year", "month"] if c in df.columns]

    try:
        exists = any(f.name == "_delta_log" for f in mssparkutils.fs.ls(path))
    except:
        exists = False

    if not exists:
        (df.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .partitionBy(*partition_cols)
            .save(path))
        print(f" Created {table}")
        return

    delta_table = DeltaTable.forPath(spark, path)

    (delta_table.alias("t")
        .merge(df.alias("s"), f"t.{primary_key} = s.{primary_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f" Upserted {table}")

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 11, Finished, Available, Finished, False)

In [10]:
def deduplicate(df, key, order_col):
    window = Window.partitionBy(key).orderBy(desc(order_col))
    return (df.withColumn("_rn", row_number().over(window))
              .filter(col("_rn") == 1)
              .drop("_rn"))


def apply_dq(df, rules):
    for name, expr in rules.items():
        df = df.withColumn(name, F.expr(expr))
    return df


def filter_valid(df):
    fail_cols = [c for c in df.columns if c.startswith("fail_")]

    cond = None
    for c in fail_cols:
        cond = (~col(c)) if cond is None else (cond & ~col(c))

    return df.filter(cond).drop(*fail_cols)


def get_failures(df):
    return {
        c.replace("fail_", ""): df.filter(col(c)).count()
        for c in df.columns if c.startswith("fail_")
    }

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 12, Finished, Available, Finished, False)

In [11]:
PIPELINE_CONFIG = {

    "customers": {
        "pk": "customer_id",
        "timestamp": "updated_at",
        "dq": {
            "fail_null_customer_id": "customer_id IS NULL",
            "fail_invalid_status": "UPPER(status) NOT IN ('ACTIVE','SUSPENDED','CHURNED')"
        }
    },

    "subscriptions": {
        "pk": "sub_id",
        "timestamp": "updated_at",
        "dq": {
            "fail_null_sub_id": "sub_id IS NULL",
            "fail_invalid_spend": "monthly_spend <= 0"
        },
        "ref": ("customers", "customer_id")
    },

    "network_events": {
        "pk": "event_id",
        "timestamp": "created_at",
        "dq": {
            "fail_null_event_id": "event_id IS NULL",
            "fail_invalid_duration": "duration_secs <= 0"
        },
        "ref": ("towers", "tower_id")
    },

    "towers": {
        "pk": "tower_id",
        "timestamp": "updated_at",
        "dq": {
            "fail_null_tower_id": "tower_id IS NULL"
        }
    },

    "vendors": {
        "pk": "vendor_id",
        "timestamp": "updated_at",
        "dq": {
            "fail_null_vendor_id": "vendor_id IS NULL"
        }
    },

    "fx_rates": {
        "pk": "rate_id",
        "timestamp": "rate_date",
        "dq": {
            "fail_invalid_rate": "rate <= 0"
        }
    },

    "support_tickets": {
        "pk": "TicketID",
        "timestamp": "CreatedDate",
        "dq": {
            "fail_null_ticket_id": "TicketID IS NULL"
        }
    }
}

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 13, Finished, Available, Finished, False)

In [12]:
def process_table(table):

    print(f"\n── Processing {table}")

    cfg = PIPELINE_CONFIG[table]

    # Incremental read
    df, latest_ts = read_incremental(table, cfg["timestamp"])

    if df.count() == 0:
        print("  No new data")
        return

    total = df.count()

    # DQ checks
    df = apply_dq(df, cfg["dq"])
    failures = get_failures(df)

    for k, v in failures.items():
        print(f"  {k}: {v:,}")

    df = filter_valid(df)

    # Dedup
    df = deduplicate(df, cfg["pk"], cfg["timestamp"])

    # Referential integrity
    if "ref" in cfg:
        ref_table, ref_key = cfg["ref"]

        df_ref = spark.read.format("delta") \
            .load(f"{SILVER_BASE}/{ref_table}") \
            .select(ref_key)

        df = df.join(df_ref, on=ref_key, how="inner")

    passed = df.count()

    # Write
    write_silver(df, table, cfg["pk"])

    # Update watermark
    update_watermark(WATERMARK_PATHS[table], latest_ts)

    # Ledger
    log_to_ledger("bronze", table, total, total, 0)
    log_to_ledger("silver", table, total, passed, total - passed)

    dq_log = build_dq_log(
    table=table,
    total=total,
    passed=passed,
    failure_reasons=failures,
    latest_ts=latest_ts
)

write_dq_log(table, dq_log)

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 14, Finished, Available, Finished, False)

In [13]:
for table in PIPELINE_CONFIG.keys():
    process_table(table)

print("PIPELINE COMPLETE")

StatementMeta(, 290e8beb-379d-455e-a191-4589fd2c2d00, 15, Finished, Available, Finished, False)


── Processing customers
  No new data

── Processing subscriptions
  No new data

── Processing network_events
  No new data

── Processing towers
  No new data

── Processing vendors
  No new data

── Processing fx_rates
  No new data

── Processing support_tickets
  No new data
PIPELINE COMPLETE
